# Teacher trace generation for SATBench (Qwen3.6-35B-A3B)

This notebook generates supervised fine-tuning data for the COS 598B project: for
each row of `satbench_with_certificates_full.jsonl`, it asks the teacher
**Qwen/Qwen3.6-35B-A3B** to solve the puzzle, anchored to the Z3 certificate, and
saves the full prompt + thinking trace + final answer to one JSON file per row.

The notebook is laid out so each cell can be run and validated independently:

1. Imports
2. System prompt
3. DIMACS → structural variable conversion (with a sanity check)
4. Reshape assignment helper (with a sanity check)
5. Certificate rendering (with a sanity check)
6. User-prompt assembly (with a full preview on a real row)
7. Response parsing + I/O helpers (with a sanity check)
8. **Configuration** — paths, sampling params, index range
9. Load dataset (with a preview)
10. Load tokenizer + model
11. Chat-template renderer
12. **Single-row test** — run the teacher on one example before launching the loop
13. Main loop over the dataset (resumable)

> **Throughput note.** This script uses `transformers.generate`, which is
> the cleanest way to validate prompts and the model end-to-end but is slow
> for full-dataset runs (a 35B-A3B MoE on a single A100 80B will take days
> to cover all 2,100 rows at `max_new_tokens=32768`). For the production
> 2,100-row run, swap the generation cell for a vLLM or sglang server using
> the same `SYSTEM_PROMPT` / `build_user_prompt` helpers — the prompt
> contract is decoupled from the generation backend on purpose.


## 1. Imports

In [1]:
from __future__ import annotations

import json
import os
import re
import sys
import time
from pathlib import Path
from typing import Any, Dict, List, Optional, Sequence, Tuple

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

print("torch          :", torch.__version__)
print("CUDA available :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("CUDA device    :", torch.cuda.get_device_name(0))
    print("GPU memory (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))


/scratch/network/yd1202/COS598B-project/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


torch          : 2.11.0+cu130
CUDA available : True
CUDA device    : NVIDIA A100 80GB PCIe
GPU memory (GB): 85.1


## 2. System prompt

This mirrors the spirit of the SATBench evaluation prompt (Figure A3):
constraints come only from `<conditions>`, variables are independent, unmentioned
variables are irrelevant. The "ground-truth certificate" framing is a
distillation-only addition; the teacher is told to **derive**, not copy.

In [2]:
SYSTEM_PROMPT = """You are an expert in propositional logic and Boolean
satisfiability (SAT). You are helping to generate high-quality reasoning
traces that a smaller student model will be fine-tuned on.

You will be given:
  * a natural-language logic puzzle (scenario + variable mapping +
    conditions + question), and
  * the underlying CNF formula (dims, num_vars, num_clauses, the raw
    DIMACS clauses, and the human-readable formula), and
  * a solver-verified Z3 certificate (a satisfying assignment for SAT
    instances, or an unsat core for UNSAT instances).

Reasoning rules (these match the SATBench evaluation protocol):
  * All constraints come ONLY from the <conditions> section. The
    <scenario> provides background and intuition, but does not impose
    any additional rules.
  * Variables represent INDEPENDENT decisions. Do not assume mutual
    exclusivity, totality, or any commonsense linkage unless the
    <conditions> state it explicitly.
  * Variables not mentioned in any condition are unconstrained and
    irrelevant to satisfiability; they may be set arbitrarily.

Task:
  1. Reason step by step. Map every natural-language condition to the
     underlying boolean variables, and either build a satisfying
     assignment by backtracking search or derive a contradiction by
     unit propagation.
  2. Use the provided Z3 certificate to anchor your reasoning, but you
     must SHOW the derivation. Do not simply restate the certificate.
  3. End your final answer with the SATBench evaluation format:
       - For SAT: a structured assignment matching `dims`, followed
         by the literal tag [SAT].
       - For UNSAT: a list of the conditions that together form the
         contradiction, followed by the literal tag [UNSAT].

Output structure (the <think> block becomes the training-time chain
of thought; the rest is the final answer):

<think>
... full step-by-step reasoning ...
</think>

Decision: <SAT or UNSAT>

Certificate:
  For SAT, provide a structured assignment in this exact form
  (a nested list whose shape equals `dims`, with 1 = True, 0 = False):
      Assignment: [[...], [...], ...]
  For UNSAT, list the natural-language condition numbers (1-indexed)
  that together cannot all be satisfied, e.g.:
      UNSAT core (condition numbers): [1, 3, 4]

Explanation:
  A short justification (2-5 sentences) that references the key
  conditions by their natural-language number.

[SAT] or [UNSAT]
"""

print(SYSTEM_PROMPT[:400], "...")


You are an expert in propositional logic and Boolean
satisfiability (SAT). You are helping to generate high-quality reasoning
traces that a smaller student model will be fine-tuned on.

You will be given:
  * a natural-language logic puzzle (scenario + variable mapping +
    conditions + question), and
  * the underlying CNF formula (dims, num_vars, num_clauses, the raw
    DIMACS clauses, and the ...


## 3. DIMACS → structural variable conversion

The dataset stores clauses in 1-indexed flat DIMACS form (`[1, -3]`) but the
natural-language puzzle and `readable` formula use the structural form
`x(i,)` / `x(i, j)` / `x(i, j, k)` derived from `dims`. The helpers below
convert between them, row-major (C-order).

In [3]:
def dimacs_to_struct(var_id: int, dims: Sequence[int]) -> str:
    """1-indexed DIMACS variable id -> structural form like 'x(2, 2, 0)'."""
    if not dims:
        return f"x{var_id}"
    flat = var_id - 1
    indices: List[int] = []
    n = len(dims)
    for k in range(n):
        stride = 1
        for d in dims[k + 1:]:
            stride *= d
        indices.append(flat // stride)
        flat = flat % stride
    if n == 1:
        # SATBench writes 1-D variables as 'x(0,)' (Python tuple style).
        return f"x({indices[0]},)"
    return f"x({', '.join(str(i) for i in indices)})"


def literal_to_struct(lit: int, dims: Sequence[int]) -> str:
    sign = "¬" if lit < 0 else ""
    return f"{sign}{dimacs_to_struct(abs(lit), dims)}"


def clause_to_struct(clause: Sequence[int], dims: Sequence[int]) -> str:
    return "(" + " ∨ ".join(literal_to_struct(l, dims) for l in clause) + ")"


**Sanity check** against the example rows:

* `dims=[5]`, var 1 → `x(0,)`, var 3 → `x(2,)`
* `dims=[3, 5, 2]`, var 25 → `x(2, 2, 0)` ("Wonder Woman uses agility in the city")
* clause `[-1, 3]` in `dims=[5]` → `(¬x(0,) ∨ x(2,))`, which matches `readable[0]`.

In [4]:
print("=== Example 1 (dims=[5]) ===")
print("  var 1            ->", dimacs_to_struct(1, [5]),  "  (expect x(0,))")
print("  var 3            ->", dimacs_to_struct(3, [5]),  "  (expect x(2,))")
print("  clause [-1, 3]   ->", clause_to_struct([-1, 3], [5]),
      "  (expect (¬x(0,) ∨ x(2,)))")

print("\n=== Example 6 (dims=[3, 5, 2]) ===")
for v in [1, 13, 18, 25]:
    print(f"  var {v:>2}           -> {dimacs_to_struct(v, [3, 5, 2])}")
print("  clause [-13, 18] ->", clause_to_struct([-13, 18], [3, 5, 2]),
      "  (expect (¬x(1, 1, 0) ∨ x(1, 3, 1)))")


=== Example 1 (dims=[5]) ===
  var 1            -> x(0,)   (expect x(0,))
  var 3            -> x(2,)   (expect x(2,))
  clause [-1, 3]   -> (¬x(0,) ∨ x(2,))   (expect (¬x(0,) ∨ x(2,)))

=== Example 6 (dims=[3, 5, 2]) ===
  var  1           -> x(0, 0, 0)
  var 13           -> x(1, 1, 0)
  var 18           -> x(1, 3, 1)
  var 25           -> x(2, 2, 0)
  clause [-13, 18] -> (¬x(1, 1, 0) ∨ x(1, 3, 1))   (expect (¬x(1, 1, 0) ∨ x(1, 3, 1)))


## 4. Reshape assignment

Z3's `sat_assignment` is a flat dict like `{"1": False, ..., "25": True, ...}`.
SATBench's trace-evaluation prompt (Figure A4) expects assignments in nested-array
form matching `dims`, e.g. `[[...], [...], ...]`. This helper converts the dict to
that structured array (1 = True, 0 = False).

In [5]:
def reshape_assignment(assignment: Dict[Any, Any],
                       dims: Sequence[int]) -> Any:
    if not dims:
        return None
    total = 1
    for d in dims:
        total *= d

    flat: List[int] = []
    for i in range(1, total + 1):
        v = assignment.get(str(i))
        if v is None:
            v = assignment.get(i, False)
        flat.append(1 if v else 0)

    def _reshape(values: List[int], shape: Sequence[int]) -> Any:
        if len(shape) == 1:
            return list(values[: shape[0]])
        d, rest = shape[0], shape[1:]
        chunk = 1
        for r in rest:
            chunk *= r
        return [_reshape(values[i * chunk:(i + 1) * chunk], rest)
                for i in range(d)]

    return _reshape(flat, list(dims))


In [6]:
# Sanity check: SAT row from example 6, only var 25 is True.
assignment = {str(i): False for i in range(1, 31)}
assignment["25"] = True
arr = reshape_assignment(assignment, [3, 5, 2])
print("Reshaped 3x5x2 array (only x(2,2,0) is True):")
for i, plane in enumerate(arr):
    print(f"  i={i}: {plane}")
print("arr[2][2][0] =", arr[2][2][0], "(expect 1)")
total_true = sum(c for plane in arr for row in plane for c in row)
print("total True cells =", total_true, "(expect 1)")


Reshaped 3x5x2 array (only x(2,2,0) is True):
  i=0: [[0, 0], [0, 0], [0, 0], [0, 0], [0, 0]]
  i=1: [[0, 0], [0, 0], [0, 0], [0, 0], [0, 0]]
  i=2: [[0, 0], [0, 0], [1, 0], [0, 0], [0, 0]]
arr[2][2][0] = 1 (expect 1)
total True cells = 1 (expect 1)


## 5. Certificate rendering

These render the Z3 certificate in a human-friendly form for the prompt:

* `render_sat_certificate` — per-variable list in structural form, plus the
  reshaped `Assignment: [[...], ...]` block that matches Figure A4.
* `render_unsat_certificate` — each `clauses[i]` rendered in DIMACS *and*
  structural form, with the solver's `unsat_reason`.

In [7]:
def render_sat_certificate(row: Dict[str, Any]) -> str:
    assignment = row.get("sat_assignment") or {}
    dims = row.get("dims") or []
    if not assignment:
        return "Certificate type: SAT (assignment is empty)"

    try:
        items = sorted(assignment.items(), key=lambda kv: int(kv[0]))
    except Exception:
        items = list(assignment.items())

    lines = []
    for var_id, val in items:
        try:
            struct = dimacs_to_struct(int(var_id), dims)
        except Exception:
            struct = f"x{var_id}"
        lines.append(f"  {struct} = {bool(val)}")
    flat_block = "\n".join(lines)

    structured = reshape_assignment(assignment, dims)
    structured_str = json.dumps(structured)

    sat_reason = row.get("sat_reason")
    extra = f"\nSolver note: {sat_reason}" if sat_reason else ""

    return (
        "Certificate type: SAT (satisfying assignment from Z3)\n"
        "Per-variable assignment (structural form):\n"
        f"{flat_block}\n"
        "Same assignment as a structured array matching `dims` "
        "(1 = True, 0 = False; this is the SATBench Assignment format):\n"
        f"  Assignment: {structured_str}{extra}"
    )


def render_unsat_certificate(row: Dict[str, Any]) -> str:
    core_indices: List[int] = row.get("unsat_core_clause_indices") or []
    clauses: List[List[int]] = row.get("clauses") or []
    dims = row.get("dims") or []

    lines = []
    for i in core_indices:
        if 0 <= i < len(clauses):
            cl = clauses[i]
            lines.append(
                f"  clauses[{i}] (DIMACS): {cl}"
                f"  ->  {clause_to_struct(cl, dims)}"
            )
        else:
            lines.append(f"  clauses[{i}]: <out of range>")
    core_block = "\n".join(lines) if lines else "  (empty)"

    unsat_reason = row.get("unsat_reason") or "(not provided)"

    return (
        "Certificate type: UNSAT (unsat core from Z3)\n"
        "UNSAT core clause indices (0-indexed into the formal `clauses` "
        f"list): {core_indices}\n"
        "These clauses, in structural form (match them to natural-language "
        "conditions by literal content, NOT by index — see ordering note "
        "below):\n"
        f"{core_block}\n"
        f"Solver-derived UNSAT reason: {unsat_reason}"
    )


def build_certificate_block(row: Dict[str, Any]) -> str:
    cert_type = row.get("certificate_type")
    if cert_type == "sat_assignment":
        return render_sat_certificate(row)
    if cert_type == "unsat_core":
        return render_unsat_certificate(row)
    return f"Certificate type: {cert_type!r} (no structured certificate)"


In [8]:
# Sanity check on a hand-crafted UNSAT row (matches example 2 in the snippet file).
unsat_row = {
    "dims": [5],
    "num_vars": 5,
    "num_clauses": 4,
    "clauses": [[3, 4], [-3, -4], [-3], [-4]],
    "certificate_type": "unsat_core",
    "unsat_core_clause_indices": [0, 2, 3],
    "unsat_reason": "Exactly-one conflict.",
}
print(build_certificate_block(unsat_row))
print()
print("---")
print()

# Sanity check on a hand-crafted SAT row (matches example 6).
sat_row = {
    "dims": [3, 5, 2],
    "num_vars": 30,
    "certificate_type": "sat_assignment",
    "sat_assignment": {str(i): (i == 25) for i in range(1, 31)},
    "sat_reason": "After removing one clause it becomes SAT.",
}
print(build_certificate_block(sat_row))


Certificate type: UNSAT (unsat core from Z3)
UNSAT core clause indices (0-indexed into the formal `clauses` list): [0, 2, 3]
These clauses, in structural form (match them to natural-language conditions by literal content, NOT by index — see ordering note below):
  clauses[0] (DIMACS): [3, 4]  ->  (x(2,) ∨ x(3,))
  clauses[2] (DIMACS): [-3]  ->  (¬x(2,))
  clauses[3] (DIMACS): [-4]  ->  (¬x(3,))
Solver-derived UNSAT reason: Exactly-one conflict.

---

Certificate type: SAT (satisfying assignment from Z3)
Per-variable assignment (structural form):
  x(0, 0, 0) = False
  x(0, 0, 1) = False
  x(0, 1, 0) = False
  x(0, 1, 1) = False
  x(0, 2, 0) = False
  x(0, 2, 1) = False
  x(0, 3, 0) = False
  x(0, 3, 1) = False
  x(0, 4, 0) = False
  x(0, 4, 1) = False
  x(1, 0, 0) = False
  x(1, 0, 1) = False
  x(1, 1, 0) = False
  x(1, 1, 1) = False
  x(1, 2, 0) = False
  x(1, 2, 1) = False
  x(1, 3, 0) = False
  x(1, 3, 1) = False
  x(1, 4, 0) = False
  x(1, 4, 1) = False
  x(2, 0, 0) = False
  x(2, 

## 6. User prompt assembly

Splits the prompt into a "Puzzle" section (what a SATBench-evaluated student
would see) and an "Auxiliary information" section (formal CNF + solver
certificate, present only to anchor the teacher's CoT).

The prompt also explicitly notes that the order of `clauses` (DIMACS list) does
**not** match the order of `Conditions` / `Readable CNF` — verified empirically
on every example in the snippet file.

In [9]:
def build_user_prompt(row: Dict[str, Any]) -> str:
    conditions = row.get("conditions") or []
    conditions_block = "\n".join(conditions) if conditions else "(none)"

    sat_label = "SAT" if row.get("satisfiable") else "UNSAT"

    return (
        "## Puzzle (this is what the evaluated student model will see)\n\n"
        f"### Scenario\n{row.get('scenario', '')}\n\n"
        f"### Variable Mapping\n{row.get('variable_mapping', '')}\n\n"
        f"### Conditions (1-indexed; this is the canonical natural-language "
        f"order)\n{conditions_block}\n\n"
        f"### Question\n{row.get('question', '')}\n\n"
        "## Auxiliary information (NOT shown at evaluation time; use to "
        "ground your reasoning)\n\n"
        "### Underlying CNF formula\n"
        f"- dims: {row.get('dims', [])}\n"
        f"- num_vars: {row.get('num_vars', 0)}\n"
        f"- num_clauses: {row.get('num_clauses', 0)}\n"
        f"- clauses (DIMACS-style; 1-indexed flat variable IDs; negative "
        f"= negated literal): {row.get('clauses', [])}\n"
        f"- Readable CNF (this list IS in the same order as `Conditions` "
        f"above): {row.get('readable', '')}\n\n"
        "### Ordering note (important)\n"
        "The order of entries in `clauses` (the DIMACS list) does NOT "
        "in general match the order of `Conditions` / `Readable CNF`. "
        "When you cite a condition number in your final answer, use the "
        "1-indexed `Conditions` order (which matches `Readable CNF`). "
        "Match `clauses[i]` to a condition by comparing literal contents, "
        "not by index.\n\n"
        f"### Ground-truth satisfiability (from Z3)\n{sat_label}\n\n"
        "### Solver-verified certificate (DERIVE this, do not copy it)\n"
        f"{build_certificate_block(row)}\n\n"
        "Now solve the puzzle. Show the full chain of thought inside a "
        "single <think> ... </think> block, then output Decision / "
        "Certificate / Explanation in the structure given by the system "
        "prompt, ending with the literal tag [SAT] or [UNSAT]."
    )


In [10]:
# Preview the full user prompt on a real-shaped row (example 6 from the snippet).
example_row = {
    "dims": [3, 5, 2],
    "num_vars": 30,
    "num_clauses": 4,
    "clauses": [[25], [-1, 13], [-13, 18], [-18]],
    "readable": "(¬x(0, 0, 0) ∨ x(1, 1, 0)) ∧ (¬x(1, 3, 1)) ∧ (¬x(1, 1, 0) ∨ x(1, 3, 1)) ∧ (x(2, 2, 0))",
    "satisfiable": True,
    "scenario": ("Three superheroes—Flash, Thor, and Wonder Woman—are engaged in missions "
                 "across two locations (0=city, 1=mountains) and four abilities "
                 "(0=speed, 1=strength, 2=agility, 3=flight)."),
    "variable_mapping": ("Let x(i, j, k) mean superhero i uses ability j at location k. "
                         "Superhero 0 is Flash, 1 is Thor, 2 is Wonder Woman."),
    "conditions": [
        "1. Either Flash does not use speed in the city, or Thor uses strength in the city.",
        "2. Thor does not use flight in the mountains.",
        "3. Either Thor does not use strength in the city, or Thor uses flight in the mountains.",
        "4. Wonder Woman uses agility in the city.",
    ],
    "question": "Can all these conditions be satisfied simultaneously?",
    "certificate_type": "sat_assignment",
    "sat_assignment": {str(i): (i == 25) for i in range(1, 31)},
    "sat_reason": "After removing one clause it becomes SAT.",
}

prompt = build_user_prompt(example_row)
print(prompt)


## Puzzle (this is what the evaluated student model will see)

### Scenario
Three superheroes—Flash, Thor, and Wonder Woman—are engaged in missions across two locations (0=city, 1=mountains) and four abilities (0=speed, 1=strength, 2=agility, 3=flight).

### Variable Mapping
Let x(i, j, k) mean superhero i uses ability j at location k. Superhero 0 is Flash, 1 is Thor, 2 is Wonder Woman.

### Conditions (1-indexed; this is the canonical natural-language order)
1. Either Flash does not use speed in the city, or Thor uses strength in the city.
2. Thor does not use flight in the mountains.
3. Either Thor does not use strength in the city, or Thor uses flight in the mountains.
4. Wonder Woman uses agility in the city.

### Question
Can all these conditions be satisfied simultaneously?

## Auxiliary information (NOT shown at evaluation time; use to ground your reasoning)

### Underlying CNF formula
- dims: [3, 5, 2]
- num_vars: 30
- num_clauses: 4
- clauses (DIMACS-style; 1-indexed flat vari

## 7. Response parsing + I/O helpers

`atomic_write_json` builds the temp path with `with_name`, not
`with_suffix(".json.tmp")` — the latter is rejected on some older Python
versions because of the multiple dots.

In [11]:
_THINK_RE = re.compile(r"<think>(.*?)</think>", flags=re.DOTALL)


def split_think(text: str) -> Tuple[Optional[str], str]:
    """Return (thinking_trace, final_answer)."""
    m = _THINK_RE.search(text)
    if m:
        thinking = m.group(1).strip()
        final = (text[: m.start()] + text[m.end():]).strip()
        return thinking, final
    if "</think>" in text:
        # Some chat templates pre-open <think> in the prompt, so the model only
        # emits the closing tag.
        j = text.index("</think>")
        return text[:j].strip(), text[j + len("</think>"):].strip()
    return None, text.strip()


def load_dataset(path: Path) -> List[Dict[str, Any]]:
    rows: List[Dict[str, Any]] = []
    with path.open("r", encoding="utf-8") as f:
        for line_no, line in enumerate(f, 1):
            line = line.strip()
            if not line:
                continue
            try:
                rows.append(json.loads(line))
            except json.JSONDecodeError as e:
                print(f"[warn] could not parse line {line_no}: {e}", file=sys.stderr)
    return rows


def already_done(out_path: Path) -> bool:
    if not out_path.exists():
        return False
    try:
        with out_path.open("r", encoding="utf-8") as f:
            data = json.load(f)
        return bool(data.get("response"))
    except Exception:
        return False


def atomic_write_json(path: Path, payload: Dict[str, Any]) -> None:
    # Use with_name() rather than with_suffix(); pre-3.13 Python rejects
    # compound suffixes like '.json.tmp' on some platforms.
    tmp = path.with_name(path.name + ".tmp")
    with tmp.open("w", encoding="utf-8") as f:
        json.dump(payload, f, ensure_ascii=False, indent=2)
    tmp.replace(path)


In [13]:
# Sanity check for split_think.
sample = "<think>let me think...\nstep 1, step 2.</think>\nDecision: SAT\n[SAT]"
trace, final = split_think(sample)
print("trace :", repr(trace))
print("final :", repr(final))

# Closing-tag-only case (Qwen3-thinking prompts pre-open <think>).
sample2 = "step 1, step 2.</think>\nDecision: UNSAT\n[UNSAT]"
trace2, final2 = split_think(sample2)
print("trace2:", repr(trace2))
print("final2:", repr(final2))

# No-thinking case.
sample3 = "Just an answer with no thinking.\n[SAT]"
trace3, final3 = split_think(sample3)
print("trace3:", repr(trace3))
print("final3:", repr(final3))


trace : 'let me think...\nstep 1, step 2.'
final : 'Decision: SAT\n[SAT]'
trace2: 'step 1, step 2.'
final2: 'Decision: UNSAT\n[UNSAT]'
trace3: None
final3: 'Just an answer with no thinking.\n[SAT]'


## 8. Configuration

Edit these to point at your data, choose the index range, and tune sampling.
Keep `END_IDX = None` to process the whole file. To shard across GPUs, run
multiple notebook copies with disjoint `[START_IDX, END_IDX)` ranges sharing
the same `OUTPUT_DIR` (resumability handles overlap).

The sampling defaults follow Qwen team's recommendation for thinking models
(temperature 0.6–0.7, top_p 0.95, top_k 20, NO repetition penalty). Adding
a repetition penalty above 1.0 degrades chain-of-thought quality because
reasoning traces naturally repeat tokens (variable names, condition labels).

In [14]:
DATASET_PATH = Path("satbench_with_certificates_full.jsonl")
OUTPUT_DIR   = Path("teacher_traces")
MODEL_ID     = "Qwen/Qwen3.6-35B-A3B"

DTYPE              = torch.bfloat16
DEVICE_MAP         = "auto"
TRUST_REMOTE_CODE  = True

MAX_NEW_TOKENS     = 32768
TEMPERATURE        = 0.7
TOP_P              = 0.95
TOP_K              = 20      # Qwen3-thinking recommended default
REPETITION_PENALTY = 1.0     # 1.0 == disabled (Qwen3 docs explicitly recommend this)
ENABLE_THINKING    = True

START_IDX = 0
END_IDX   = None        # None means "to end of dataset"
SEED      = 42
LOG_EVERY = 5

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
torch.manual_seed(SEED)
print("output_dir =", OUTPUT_DIR.resolve())


output_dir = /scratch/network/yd1202/COS598B-project/teacher_traces


## 9. Load dataset

In [15]:
rows = load_dataset(DATASET_PATH)
print(f"Loaded {len(rows)} rows from {DATASET_PATH}")
_preview_end = END_IDX if END_IDX is not None else len(rows)
print(f"Will process indices [{START_IDX}, {_preview_end}) when the main loop runs.")
print("First row keys:", sorted(rows[0].keys()))
print("First row dims / num_vars / num_clauses / satisfiable:",
      rows[0].get("dims"), rows[0].get("num_vars"),
      rows[0].get("num_clauses"), rows[0].get("satisfiable"))


Loaded 2100 rows from satbench_with_certificates_full.jsonl
Will process indices [0, 2100) when the main loop runs.
First row keys: ['certificate_type', 'clauses', 'conditions', 'consistency_check_trace_history', 'dims', 'formula_equivalence_check', 'num_clauses', 'num_vars', 'question', 'readable', 'recovered_formula', 'recovered_formula_full_text', 'sat_assignment', 'sat_reason', 'satisfiable', 'scenario', 'unsat_core_clause_indices', 'unsat_reason', 'variable_mapping']
First row dims / num_vars / num_clauses / satisfiable: [5] 5 4 False


## 10. Load tokenizer + model

This loads ~70 GB of weights in bf16 and takes a few minutes. If your machine
has 2× 40 GB cards it will shard automatically because of `device_map="auto"`.

The `pad = eos` assignment is the standard workaround for Qwen-family
tokenizers when they don't ship an explicit pad token; it's safe because the
loop runs at batch size 1, so padding never actually happens.

In [16]:
print(f"Loading tokenizer: {MODEL_ID}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=TRUST_REMOTE_CODE)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token
print("Tokenizer ready. Vocab size:", len(tokenizer))

print(f"Loading model: {MODEL_ID}")
t0 = time.time()
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=DTYPE,
    device_map=DEVICE_MAP,
    trust_remote_code=TRUST_REMOTE_CODE,
)
model.eval()
print(f"Model ready in {time.time() - t0:.0f}s")
if hasattr(model, "hf_device_map"):
    print("Device map (first 5 entries):")
    for i, (mod, dev) in enumerate(list(model.hf_device_map.items())[:5]):
        print(f"  {mod} -> {dev}")


Loading tokenizer: Qwen/Qwen3.6-35B-A3B


`torch_dtype` is deprecated! Use `dtype` instead!


Tokenizer ready. Vocab size: 248077
Loading model: Qwen/Qwen3.6-35B-A3B


Fetching 26 files: 100%|██████████| 26/26 [00:00<00:00, 115.69it/s]
The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d
Loading weights: 100%|██████████| 693/693 [00:55<00:00, 12.42it/s]


Model ready in 59s
Device map (first 5 entries):
  model.embed_tokens -> 0
  model.layers.0 -> 0
  model.layers.1 -> 0
  model.layers.2 -> 0
  model.layers.3 -> 0


## 11. Chat-template renderer

In [17]:
def render_prompt(tokenizer, system_prompt: str, user_prompt: str,
                  enable_thinking: bool) -> str:
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user",   "content": user_prompt},
    ]
    try:
        return tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=enable_thinking,
        )
    except TypeError:
        # Fallback for templates that don't accept enable_thinking.
        return tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )


## 12. Single-row test (run this BEFORE the full loop)

Pick any index, render the prompt, generate a response, and inspect it. The
expected output is:

* a non-empty `thinking_trace`,
* a `final_answer` ending in `[SAT]` or `[UNSAT]`, matching the dataset's
  `satisfiable` field,
* an `Assignment: [...]` block (for SAT) whose shape equals `dims`, or a
  list of condition numbers (for UNSAT).

If anything looks off, edit the prompt cells above and rerun this cell — the
model is already loaded so each test only costs one generation.

In [18]:
TEST_IDX = 0   # try a SAT and an UNSAT row
test_row = rows[TEST_IDX]

user_prompt = build_user_prompt(test_row)
prompt_text = render_prompt(tokenizer, SYSTEM_PROMPT, user_prompt, ENABLE_THINKING)

print("satisfiable (ground truth):", test_row.get("satisfiable"))
print("certificate_type:",          test_row.get("certificate_type"))
print()
print("===== Rendered prompt (first 1500 chars) =====")
print(prompt_text[:1500])
print("...")
print()

inputs = tokenizer(prompt_text, return_tensors="pt").to(model.device)
prompt_len = int(inputs["input_ids"].shape[1])
print(f"prompt_len = {prompt_len} tokens")
print("Generating ...")

t_start = time.time()
with torch.no_grad():
    output_ids = model.generate(
        **inputs,
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=True,
        temperature=TEMPERATURE,
        top_p=TOP_P,
        top_k=TOP_K,
        repetition_penalty=REPETITION_PENALTY,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )
elapsed = time.time() - t_start
gen_ids = output_ids[0, prompt_len:]
full_response = tokenizer.decode(gen_ids, skip_special_tokens=True)
thinking_trace, final_answer = split_think(full_response)

print(f"\nGenerated {gen_ids.shape[0]} tokens in {elapsed:.1f}s "
      f"({gen_ids.shape[0] / max(elapsed, 1e-6):.1f} tok/s)")

print("\n===== Thinking trace (first 1500 chars) =====")
print((thinking_trace or "(none extracted)")[:1500])

print("\n===== Final answer =====")
print(final_answer)


satisfiable (ground truth): False
certificate_type: unsat_core

===== Rendered prompt (first 1500 chars) =====
<|im_start|>system
You are an expert in propositional logic and Boolean
satisfiability (SAT). You are helping to generate high-quality reasoning
traces that a smaller student model will be fine-tuned on.

You will be given:
  * a natural-language logic puzzle (scenario + variable mapping +
    conditions + question), and
  * the underlying CNF formula (dims, num_vars, num_clauses, the raw
    DIMACS clauses, and the human-readable formula), and
  * a solver-verified Z3 certificate (a satisfying assignment for SAT
    instances, or an unsat core for UNSAT instances).

Reasoning rules (these match the SATBench evaluation protocol):
  * All constraints come ONLY from the <conditions> section. The
    <scenario> provides background and intuition, but does not impose
    any additional rules.
  * Variables represent INDEPENDENT decisions. Do not assume mutual
    exclusivity, total

## 13. Main loop

Resumable: any output JSON that already has a non-empty `response` field is
skipped. To re-run a specific index, delete its JSON in `OUTPUT_DIR` first.

Each saved record carries an OpenAI-style `messages` field for direct use
with TRL's `SFTTrainer` / axolotl, plus the rich metadata (raw prompt parts,
generation kwargs, separated `thinking_trace` / `final_answer`, original
SATBench row) for inspection and filtering.

In [ ]:
# Recompute the iteration range here so changes to START_IDX/END_IDX in the
# config cell take effect even if cell 9 wasn't re-run.
end_idx = END_IDX if END_IDX is not None else len(rows)

n_done = n_skipped = n_failed = 0
t0 = time.time()

for idx in range(START_IDX, end_idx):
    row = rows[idx]
    out_path = OUTPUT_DIR / f"satbench_{idx:05d}.json"

    if already_done(out_path):
        n_skipped += 1
        continue

    user_prompt = build_user_prompt(row)
    prompt_text = render_prompt(tokenizer, SYSTEM_PROMPT, user_prompt, ENABLE_THINKING)

    inputs = tokenizer(prompt_text, return_tensors="pt").to(model.device)
    prompt_len = int(inputs["input_ids"].shape[1])

    try:
        with torch.no_grad():
            output_ids = model.generate(
                **inputs,
                max_new_tokens=MAX_NEW_TOKENS,
                do_sample=True,
                temperature=TEMPERATURE,
                top_p=TOP_P,
                top_k=TOP_K,
                repetition_penalty=REPETITION_PENALTY,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id,
            )
    except torch.cuda.OutOfMemoryError as e:
        print(f"[oom] idx={idx}: {e}", file=sys.stderr)
        torch.cuda.empty_cache()
        n_failed += 1
        continue
    except Exception as e:
        print(f"[error] generation failed on idx={idx}: {e}", file=sys.stderr)
        n_failed += 1
        continue

    gen_ids = output_ids[0, prompt_len:]
    full_response = tokenizer.decode(gen_ids, skip_special_tokens=True)
    thinking_trace, final_answer = split_think(full_response)

    record = {
        "index": idx,
        "model_id": MODEL_ID,
        "satbench_row": row,
        # OpenAI-style messages for direct SFTTrainer / axolotl consumption.
        # The assistant turn is the FULL response, including any <think> block.
        "messages": [
            {"role": "system",    "content": SYSTEM_PROMPT},
            {"role": "user",      "content": user_prompt},
            {"role": "assistant", "content": full_response},
        ],
        # Rich metadata kept alongside for filtering / debugging.
        "prompt": {
            "system": SYSTEM_PROMPT,
            "user": user_prompt,
            "rendered": prompt_text,
        },
        "generation_kwargs": {
            "max_new_tokens":     MAX_NEW_TOKENS,
            "temperature":        TEMPERATURE,
            "top_p":              TOP_P,
            "top_k":              TOP_K,
            "repetition_penalty": REPETITION_PENALTY,
            "do_sample":          True,
            "enable_thinking":    ENABLE_THINKING,
        },
        "response":              full_response,
        "thinking_trace":        thinking_trace,
        "final_answer":          final_answer,
        "num_prompt_tokens":     prompt_len,
        "num_generated_tokens":  int(gen_ids.shape[0]),
    }
    atomic_write_json(out_path, record)

    n_done += 1
    if n_done % LOG_EVERY == 0:
        elapsed = time.time() - t0
        rate = n_done / max(elapsed, 1e-6)
        print(f"[info] done={n_done} skipped={n_skipped} failed={n_failed} "
              f"rate={rate:.2f}/s elapsed={elapsed:.0f}s")

print(f"[done] done={n_done} skipped={n_skipped} failed={n_failed} "
      f"total_time={time.time() - t0:.0f}s -> {OUTPUT_DIR}")


In [1]:
import json
import re
from pathlib import Path
from typing import Dict, Any

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer


/scratch/network/yd1202/COS598B-project/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
MODEL_NAME = "Qwen/Qwen3.5-4B"

SYSTEM_PROMPT = "You are a helpful assistant. Think carefully before answering."

# Thinking mode often needs more tokens than normal direct-answer mode.
MAX_NEW_TOKENS = 4096

# For benchmark-style deterministic generation, keep DO_SAMPLE = False.
DO_SAMPLE = False
TEMPERATURE = 0.6
TOP_P = 0.95

# Set to True only if bitsandbytes is installed and you want lower memory use.
LOAD_IN_4BIT = False


In [3]:
def load_model_and_tokenizer(model_name: str, load_in_4bit: bool = False):
    print(f"Loading tokenizer: {model_name}")
    tokenizer = AutoTokenizer.from_pretrained(
        model_name,
        trust_remote_code=True,
    )

    print(f"Loading model: {model_name}")

    model_kwargs = {
        "device_map": "auto",
        "trust_remote_code": True,
    }

    if load_in_4bit:
        model_kwargs.update(
            {
                "load_in_4bit": True,
                "bnb_4bit_compute_dtype": torch.bfloat16,
                "bnb_4bit_use_double_quant": True,
                "bnb_4bit_quant_type": "nf4",
            }
        )
    else:
        model_kwargs["torch_dtype"] = "auto"

    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        **model_kwargs,
    )

    model.eval()
    return model, tokenizer


In [4]:
model, tokenizer = load_model_and_tokenizer("Qwen/Qwen3.5-2B", LOAD_IN_4BIT)


Loading tokenizer: Qwen/Qwen3.5-4B
Loading model: Qwen/Qwen3.5-4B


Fetching 2 files: 100%|██████████| 2/2 [01:10<00:00, 35.36s/it]
[transformers] The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d
Loading weights: 100%|██████████| 426/426 [00:05<00:00, 80.84it/s] 


In [5]:
def split_thinking_and_answer(text: str) -> Dict[str, str]:
    match = re.search(r"<think>(.*?)</think>(.*)", text, flags=re.DOTALL)

    if not match:
        return {
            "thinking": "",
            "answer": text.strip(),
            "raw": text.strip(),
        }

    thinking = match.group(1).strip()
    answer = match.group(2).strip()

    return {
        "thinking": thinking,
        "answer": answer,
        "raw": text.strip(),
    }


In [6]:
@torch.inference_mode()
def generate_once(
    user_prompt: str,
    system_prompt: str = SYSTEM_PROMPT,
    max_new_tokens: int = MAX_NEW_TOKENS,
    temperature: float = TEMPERATURE,
    top_p: float = TOP_P,
    do_sample: bool = DO_SAMPLE,
) -> Dict[str, str]:
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]

    # Key line: enable_thinking=True turns on Qwen thinking mode.
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=True,
    )

    inputs = tokenizer([text], return_tensors="pt").to(model.device)

    generation_kwargs = {
        "max_new_tokens": max_new_tokens,
        "pad_token_id": tokenizer.eos_token_id,
    }

    if do_sample:
        generation_kwargs.update(
            {
                "do_sample": True,
                "temperature": temperature,
                "top_p": top_p,
            }
        )
    else:
        generation_kwargs["do_sample"] = False

    output_ids = model.generate(
        **inputs,
        **generation_kwargs,
    )

    generated_ids = output_ids[0][inputs.input_ids.shape[-1]:]

    decoded = tokenizer.decode(
        generated_ids,
        skip_special_tokens=False,
    )

    return split_thinking_and_answer(decoded)


In [8]:
prompt = """
Determine whether this SATBench-style logic puzzle is SAT or UNSAT. Explain briefly.

Puzzle:
Alice is either a knight or a knave. Knights always tell the truth, and knaves always lie.
Alice says: I am a knight.
"""

import time

start = time.time()

result = generate_once(prompt)

end = time.time()
print(end - start, "s")

print(result["raw"])


185.89827799797058 s
Thinking Process:

1.  **Analyze the Request:**
    *   Task: Determine if the given logic puzzle is SAT (satisfiable) or UNSAT (unsatisfiable).
    *   Format: SATBench-style logic puzzle.
    *   Explanation: Briefly explain the reasoning.
    *   Puzzle Content:
        *   Alice is either a knight or a knave.
        *   Knights always tell the truth.
        *   Knaves always lie.
        *   Alice says: "I am a knight."

2.  **Analyze the Puzzle Logic:**
    *   Let $A$ be the proposition "Alice is a knight".
    *   Let $\neg A$ be the proposition "Alice is a knave" (since she is either a knight or a knave, these are mutually exclusive and exhaustive).
    *   Condition 1: If $A$ is true (Alice is a knight), then her statement must be true.
    *   Condition 2: If $A$ is false (Alice is a knave), then her statement must be false.
    *   Alice's Statement ($S$): "I am a knight" ($A$).
    *   Truth value of the statement depends on Alice's type.
    *   Case

In [9]:
import json

path = "dataset/satbench_teacher_prompts.jsonl"

rows = []
with open(path, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        rows.append(json.loads(line))

print(len(rows))
print(rows[0].keys())
print(rows[0]["messages"])

2100
dict_keys(['index', 'prompt_id', 'label', 'satisfiable', 'certificate_type', 'messages', 'system_prompt', 'user_prompt', 'ground_truth'])
[{'role': 'system', 'content': 'You are an expert in propositional logic and Boolean satisfiability (SAT).\nYou are generating high-quality teacher outputs for supervised fine-tuning of a smaller language model.\n\nYou will be given a SATBench-style natural-language logic puzzle, its underlying CNF formula, and a Z3-verified certificate.\n\nImportant reasoning rules:\n- Use only the constraints stated in the conditions. The scenario is background only and adds no hidden constraints.\n- Treat all variables as independent Boolean decisions unless the conditions explicitly state otherwise.\n- Do not add commonsense assumptions such as mutual exclusivity, exactly-one constraints, or real-world causal links unless they are stated in the conditions.\n- Variables not mentioned in the conditions are irrelevant to satisfiability and may be assigned arbit

In [10]:
first = rows[0]

system_prompt = first["system_prompt"]
user_prompt = first["user_prompt"]
messages = first["messages"]
label = first["label"]
ground_truth = first["ground_truth"]

print(label)
print(messages[0]["role"], messages[0]["content"][:200])
print(messages[1]["role"], messages[1]["content"][:200])

UNSAT
system You are an expert in propositional logic and Boolean satisfiability (SAT).
You are generating high-quality teacher outputs for supervised fine-tuning of a smaller language model.

You will be given a 
user ## Puzzle

<scenario>
In a quiet neighborhood, there are two stray cats—Whiskers and Midnight—that occasionally visit a friendly neighbor's porch. Each cat has the independent option to visit the porc


In [2]:
import pandas as pd
df = pd.read_json('/scratch/network/yd1202/COS598B-project/teacher_responses_qwen35_2b/records/row_00000.json', lines=True)
print(df)

ValueError: Expected object or value